# Power BI Data Preparation

This notebook prepares the processed Version 2 job market dataset for Power BI by transforming the dataset into a clean relational model.

## Objective

The goal of this notebook is to create Power BI-ready tables from the processed job postings dataset.

The output tables will be used to build a scalable dashboard data model with clear relationships between job postings and extracted skills.

In [22]:
import pandas as pd
import ast

## 1. Load the Processed Dataset

In [ ]:
from pathlib import Path

# Support launching Jupyter from the repository root or notebooks directory.
working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (working_dir, *working_dir.parents)
     if (candidate / "notebooks" / "processed_job_data.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate notebooks/processed_job_data.csv. "
        "Open this notebook from the cloned repository."
    )

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
df = pd.read_csv(NOTEBOOK_DIR / "processed_job_data.csv")
df.head()


In [24]:
for column in df.columns:
    if column.startswith('extracted_'):
        df[column] = df[column].apply(ast.literal_eval)
df.head()

,standardized_job_title,clean_job_title,company_name,job_country,job_location,employment_type,is_remote,job_description,job_source,posted_at_utc,normalized_job_description,extracted_programming_languages,extracted_bi_tools,extracted_databases,extracted_cloud,extracted_data_engineering,extracted_machine_learning,extracted_big_data,extracted_ai_technologies
0,data analyst,(usa) data analyst ii,Vizio,US,NaN,full-time,False,"Position Summary...As a Data Analyst II, you w...",ziprecruiter,2026-06-10 00:00:00+00:00,"position summary...as a data analyst ii, you w...","[python, sql, r, scala]","[power bi, tableau]",[],[],[],[],[],[]
1,data analyst,"(usa) senior, data analyst",Walmart,US,NaN,full-time,False,Position Summary...\n\nWhat you'll do...Senior...,ziprecruiter,2026-06-11 00:00:00+00:00,position summary...\n\nwhat you'll do...senior...,"[python, sql, r, scala]","[power bi, tableau]",[],[],[],[],[],[]
2,data scientist,<new graduate hiring> data scientist,NXP Semiconductors,SG,NaN,full-time,False,Data Scientist\n\nAbout the Role\n\nWe are see...,built in,2026-06-01 00:00:00+00:00,data scientist\n\nabout the role\n\nwe are see...,"[python, sql, r]","[power bi, tableau]",[],"[aws, azure]",[],[scikit-learn],[],"[machine learning, large language models, gene..."
3,business intelligence analyst,advanced business intelligence analyst waterdown,L3Harris Technologies,CA,NaN,full-time,False,Location: Northwest Newfoundland/Eastern Labra...,learn4good,2026-06-09 00:00:00+00:00,location: northwest newfoundland/eastern labra...,[sql],[power bi],[],[],[],[],[],[]
4,data analyst,afc data analyst,AtkinsRéalis,SA,NaN,full-time,False,Overview\n\nWe are seeking an experienced AFC ...,atkinsréalis careers,2026-06-09 00:00:00+00:00,overview\n\nwe are seeking an experienced afc ...,"[python, sql, r]","[power bi, tableau]",[],[],[],[],[hadoop],[machine learning]


## 2. Create the Jobs Table

In [ ]:
df['job_id'] = range(1, len(df) + 1)
excluded_columns = [column for column in df.columns if column.startswith('extracted_')]

df_jobs = df.drop(columns=excluded_columns)
df_jobs.head()

,standardized_job_title,clean_job_title,company_name,job_country,job_location,employment_type,is_remote,job_description,job_source,posted_at_utc,normalized_job_description,job_id
0,data analyst,(usa) data analyst ii,Vizio,US,NaN,full-time,False,"Position Summary...As a Data Analyst II, you w...",ziprecruiter,2026-06-10 00:00:00+00:00,"position summary...as a data analyst ii, you w...",1
1,data analyst,"(usa) senior, data analyst",Walmart,US,NaN,full-time,False,Position Summary...\n\nWhat you'll do...Senior...,ziprecruiter,2026-06-11 00:00:00+00:00,position summary...\n\nwhat you'll do...senior...,2
2,data scientist,<new graduate hiring> data scientist,NXP Semiconductors,SG,NaN,full-time,False,Data Scientist\n\nAbout the Role\n\nWe are see...,built in,2026-06-01 00:00:00+00:00,data scientist\n\nabout the role\n\nwe are see...,3
3,business intelligence analyst,advanced business intelligence analyst waterdown,L3Harris Technologies,CA,NaN,full-time,False,Location: Northwest Newfoundland/Eastern Labra...,learn4good,2026-06-09 00:00:00+00:00,location: northwest newfoundland/eastern labra...,4
4,data analyst,afc data analyst,AtkinsRéalis,SA,NaN,full-time,False,Overview\n\nWe are seeking an experienced AFC ...,atkinsréalis careers,2026-06-09 00:00:00+00:00,overview\n\nwe are seeking an experienced afc ...,5


## 3. Create the Skills Table

In [48]:
skill_records = []

extracted_columns = [
    column
    for column in df.columns
    if column.startswith("extracted_")
]

for column in extracted_columns:
    category = column.replace("extracted_", "")

    for _, row in df.iterrows():
        job_id = row["job_id"]
        skills = row[column]

        for skill in skills:
            skill_records.append({
                "job_id": job_id,
                "skill": skill,
                "category": category
            })

skills_table = pd.DataFrame(skill_records)



In [49]:
skills_table

,job_id,skill,category
0,1,python,programming_languages
1,1,sql,programming_languages
2,1,r,programming_languages
3,1,scala,programming_languages
4,2,python,programming_languages
...,...,...,...
5860,1037,machine learning,ai_technologies
5861,1037,deep learning,ai_technologies
5862,1037,nlp,ai_technologies
5863,1038,machine learning,ai_technologies


## 4. Validate Power BI Tables

Before exporting the tables, we validate the structure and quality of the Power BI data model.

The goal is to confirm that:

- `jobs_table` contains one unique row per job posting
- `skills_table` contains valid extracted skill records
- all expected skill categories are present
- no duplicate skill records were introduced
- the tables are ready for a one-to-many relationship in Power BI

In [60]:
# Validate primary key uniqueness in jobs table
print("Is job_id unique:", df_jobs["job_id"].is_unique)
print("Duplicate job_id count:", df_jobs["job_id"].duplicated().sum())

# Validate missing values in skills table
print("Missing skills:", skills_table["skill"].isnull().sum())

# Validate table dimensions
print("Jobs table shape:", df_jobs.shape)
print("Skills table shape:", skills_table.shape)

# Validate extracted skill categories
print(skills_table["category"].value_counts())

# Validate duplicate skill records
duplicate_skill_records = skills_table.duplicated(
    subset=["job_id", "skill", "category"]
).sum()

print("Duplicate skill records:", duplicate_skill_records)

Is job_id unique: True
Duplicate job_id count: 0
Missing skills: 0
Jobs table shape: (1044, 12)
Skills table shape: (5865, 3)
category
ai_technologies          1689
programming_languages    1468
cloud                     729
bi_tools                  534
machine_learning          436
data_engineering          434
databases                 313
big_data                  262
Name: count, dtype: int64
Duplicate skill records: 0


In [ ]:
df_jobs.to_csv(NOTEBOOK_DIR / "jobs_table.csv", index=False)
skills_table.to_csv(NOTEBOOK_DIR / "skills_table.csv", index=False)
